# Carta Azul: source validation

This read-only companion notebook inventories the closed Applebee's source corpus,
checks every expected daily partition, and profiles one explicitly bounded sample
using the production Silver schema and monetary expressions. It explains the
source checks that support the pipeline; it does not deploy or refresh anything.

**Provenance.** This is a curated diagnostic derived from the original
`02_pipeline_profiling` notebook and aligned with the final Silver implementation.
It is not an identical export of that earlier exploration. The current production
schema includes fractional void quantities as `DECIMAL(18,4)`.

**Execution status.** This curated notebook has no embedded execution evidence.
Run its cells in order to obtain current results. Historical pipeline acceptance
and complete-data reconciliation are documented separately in the operational
README; a one-day profile does not replace those full-history checks.


## Inputs and prerequisites

- A Databricks serverless Python notebook with an active Spark session and `dbutils`.
- Permission to read the source Unity Catalog Volume and the uploaded code files.
- The existing `src/transformations` directory in private Workspace Files. Adjust
  `CODE_ROOT` below to its location before running.
- Transaction files for every day from April 28 through August 20, 2026, and the
  `employees`, `items`, `locations`, and `revenue_centers` reference directories.

No custom Spark configuration keys are required. Only the schema module is
imported normally: importing `silver._common` directly outside a pipeline can fail
on restricted custom Spark configuration. Two pure expression helpers are loaded
from its source without executing that module's configuration code.

This notebook creates only in-memory DataFrames and aggregate outputs. It contains
no DDL, source writes, file exports, job or pipeline creation, or execution trigger.
It never displays raw tickets or employee identifiers, names, dates of birth, or
payroll attributes. Keep notebook access within the authorized project team.


In [ ]:
from datetime import date, timedelta
from decimal import Decimal
from pathlib import Path
import ast
import hashlib
import importlib
import json
import sys

from pyspark.sql import functions as F

BASE_PATH = "/Volumes/workspace/applebees/raw_files/data/Dataset"
CODE_ROOT = "/Workspace/Users/a01352461@tec.mx/CartaAzul/src/transformations"
BRAND = "rmh"
EXPECTED_START = date(2026, 4, 28)
EXPECTED_END = date(2026, 8, 20)
EXPECTED_FILE_COUNT = 115
SAMPLE_DATE = date(2026, 4, 28)
TOLERANCE = Decimal("0.05")

assert EXPECTED_START <= SAMPLE_DATE <= EXPECTED_END
expected_dates = {
    EXPECTED_START + timedelta(days=offset)
    for offset in range((EXPECTED_END - EXPECTED_START).days + 1)
}
assert len(expected_dates) == EXPECTED_FILE_COUNT

def emit(label, rows):
    """Print aggregate records only; keep Decimal values exact in JSON text."""
    values = [row.asDict(recursive=True) if hasattr(row, "asDict") else row for row in rows]
    print(label + "=" + json.dumps(values, default=str, sort_keys=True))


## Validate the complete file inventory

This section lists metadata without parsing the full transaction corpus. It tests
set equality for all 115 dates, not just the first and last dates. Every expected
daily directory must contain exactly one JSON file named `checks.json`. Additional
or missing dated partitions cause an explicit failure. Original files remain in
place and are opened read-only by later cells.


In [ ]:
checks_root = f"{BASE_PATH}/facts/checks/brand={BRAND}"
date_directories = sorted(
    [entry for entry in dbutils.fs.ls(checks_root)
     if entry.isDir() and entry.name.rstrip("/").startswith("dt=")],
    key=lambda entry: entry.name,
)
actual_dates = []
for entry in date_directories:
    value = entry.name.rstrip("/").removeprefix("dt=")
    actual_dates.append(date.fromisoformat(value))
assert len(actual_dates) == len(set(actual_dates)), "Duplicate daily partition metadata."
assert set(actual_dates) == expected_dates, (
    "Daily partition coverage mismatch: "
    f"missing={sorted(expected_dates - set(actual_dates))}; "
    f"unexpected={sorted(set(actual_dates) - expected_dates)}"
)

source_inventory = []
for directory, business_day in zip(date_directories, actual_dates):
    files = sorted(
        [entry for entry in dbutils.fs.ls(directory.path)
         if not entry.isDir() and entry.name.lower().endswith(".json")
         and not entry.name.startswith(".")],
        key=lambda entry: entry.name,
    )
    assert [entry.name for entry in files] == ["checks.json"], (
        f"Expected exactly checks.json in partition {business_day}."
    )
    assert files[0].size > 0, f"Empty transaction file for {business_day}."
    source_inventory.append({"date": business_day, "path": files[0].path, "bytes": files[0].size})
assert len(source_inventory) == EXPECTED_FILE_COUNT
emit("TRANSACTION_FILE_INVENTORY", [{
    "files": len(source_inventory), "all_expected_dates_present": True,
    "first_date": min(actual_dates), "last_date": max(actual_dates),
    "total_bytes": sum(entry["bytes"] for entry in source_inventory),
    "smallest_file_bytes": min(entry["bytes"] for entry in source_inventory),
    "largest_file_bytes": max(entry["bytes"] for entry in source_inventory),
}])

dimension_paths = {}
dimension_inventory = []
for name in ["employees", "items", "locations", "revenue_centers"]:
    files = sorted(
        [entry for entry in dbutils.fs.ls(f"{BASE_PATH}/dimensions/{name}")
         if not entry.isDir() and entry.name.lower().endswith(".json")
         and not entry.name.startswith(".")],
        key=lambda entry: entry.path,
    )
    assert files and all(entry.size > 0 for entry in files), f"Missing or empty {name} JSON."
    dimension_paths[name] = [entry.path for entry in files]
    dimension_inventory.append({"dimension": name, "json_files": len(files),
                                "total_bytes": sum(entry.size for entry in files)})
emit("REFERENCE_FILE_INVENTORY", dimension_inventory)


## Load the production contract and one bounded day

The production schema excludes employee personal and payroll fields. Monetary
values and item/void quantities use exact decimals. The reader uses `FAILFAST`:
a malformed file or incompatible declared value must be corrected at the contract
level rather than silently discarded. A source schema may intentionally omit
irrelevant fields; this is not a general detector of every possible new column.

The source daily arrays are unsplittable. `coalesce(1)` mirrors the accepted Free
Edition workaround that limits concurrent Photon JSON-reader buffers. It reduces
parallelism and is not a general performance recommendation. Only April 28 is
parsed here; full-period metadata coverage does not prove full-period data quality.


In [ ]:
code_root = Path(CODE_ROOT)
assert code_root.is_dir(), "Upload src/transformations and adjust CODE_ROOT before running."
if str(code_root) not in sys.path:
    sys.path.insert(0, str(code_root))
importlib.invalidate_caches()
import silver._schemas as source_schemas
source_schemas = importlib.reload(source_schemas)
CHECKS_SCHEMA = source_schemas.CHECKS_SCHEMA

# Execute only the actual pure helper functions, never custom Spark configuration.
helper_path = code_root / "silver" / "_common.py"
helper_tree = ast.parse(helper_path.read_text())
helper_names = {"array_money_total", "money"}
helper_nodes = [node for node in helper_tree.body
                if isinstance(node, ast.FunctionDef) and node.name in helper_names]
assert {node.name for node in helper_nodes} == helper_names
helper_namespace = {"F": F}
exec(compile(ast.fix_missing_locations(ast.Module(body=helper_nodes, type_ignores=[])),
             str(helper_path), "exec"), helper_namespace)
array_money_total = helper_namespace["array_money_total"]
money = helper_namespace["money"]
emit("SOURCE_CODE_CONTRACT", [{
    "schema_sha256": hashlib.sha256((code_root / "silver" / "_schemas.py").read_bytes()).hexdigest(),
    "helper_sha256": hashlib.sha256(helper_path.read_bytes()).hexdigest(),
    "sample_date": SAMPLE_DATE,
}])

sample_path = next(entry["path"] for entry in source_inventory if entry["date"] == SAMPLE_DATE)
raw = (
    spark.read.schema(CHECKS_SCHEMA)
    .option("multiLine", "true").option("mode", "FAILFAST")
    .option("recursiveFileLookup", "true").json(sample_path)
    .select("*", F.col("_metadata.file_path").alias("source_file"))
    .coalesce(1)
    .withColumn("brand", F.regexp_extract("source_file", r"/brand=([^/]+)/", 1))
    .withColumn("partition_date", F.expr(
        "try_cast(regexp_extract(source_file, '/dt=([^/]+)/', 1) as date)"))
    .withColumn("business_date", F.expr("try_cast(BusinessDate as date)"))
)
keys = ["brand", "partition_date", "LocationId", "Id"]


In [ ]:
structural_metrics = raw.agg(
    F.count("*").alias("sample_tickets"),
    F.countDistinct(F.struct(*keys)).alias("distinct_business_keys"),
    F.countDistinct("LocationId").alias("transaction_locations"),
    F.sum(F.when(F.col("Id").isNull() | F.col("LocationId").isNull()
                 | F.col("partition_date").isNull() | (F.length("brand") == 0), 1).otherwise(0)).alias("invalid_keys"),
    F.sum(F.when(~F.col("business_date").eqNullSafe(F.col("partition_date")), 1).otherwise(0)).alias("business_partition_mismatches"),
    *[F.sum(F.coalesce(F.array_size(name), F.lit(0))).alias(name + "_events")
      for name in ["ItemsSold", "Payments", "Discounts", "Tax", "Voids"]],
    *[F.sum(F.when(F.col(name).isNull(), 1).otherwise(0)).alias(name + "_null_arrays")
      for name in ["ItemsSold", "Payments", "Discounts", "Tax", "Voids"]],
).first().asDict()
emit("SAMPLE_STRUCTURE", [structural_metrics])
assert structural_metrics["sample_tickets"] > 0, "The chosen sample contains no tickets."
assert structural_metrics["sample_tickets"] == structural_metrics["distinct_business_keys"], "Sample contains repeated business keys."
assert structural_metrics["invalid_keys"] == 0, "Sample contains invalid business keys."
assert structural_metrics["business_partition_mismatches"] == 0, "Sample business dates do not match source partitions."


## Validate reference keys and actor relationships

The `items` catalog is inventoried above, but it is not parsed or materialized by
the first pipeline version. The three operational reference contracts are checked
below. Employee records marked `Deleted` remain historical references.

Role-resolution outputs are aggregate counts only. References join on
`employees.Id`, never on `LocationEmployeeId`. A zero or unresolved manager ID is
context for review and does not prove missing authorization. Actors and approvers
do not establish who received an employee benefit.


In [ ]:
reference_schemas = {
    "employees": source_schemas.EMPLOYEES_SCHEMA,
    "locations": source_schemas.LOCATIONS_SCHEMA,
    "revenue_centers": source_schemas.REVENUE_CENTERS_SCHEMA,
}
references = {}
reference_metrics = []
for name, schema in reference_schemas.items():
    frame = (spark.read.schema(schema).option("multiLine", "true")
             .option("mode", "FAILFAST").json(dimension_paths[name]).coalesce(1))
    references[name] = frame
    expressions = [F.count("*").alias("records"), F.count("Id").alias("non_null_keys"),
                   F.countDistinct("Id").alias("distinct_keys")]
    if name == "employees":
        expressions.append(F.sum(F.when(F.col("Deleted"), 1).otherwise(0)).alias("historically_deleted_records"))
    stats = frame.agg(*expressions).first().asDict()
    assert stats["records"] > 0 and stats["records"] == stats["non_null_keys"] == stats["distinct_keys"], (
        f"The {name} reference contains absent or repeated Id keys."
    )
    reference_metrics.append({"dimension": name, **stats})
emit("REFERENCE_KEY_QUALITY", reference_metrics)

# Each collection is expanded independently; there is no item/payment cross join.
payments = raw.select(*keys, F.posexplode("Payments").alias("payment_position", "p"))
check_discounts = raw.select(*keys, F.posexplode("Discounts").alias("discount_position", "d"))
item_rows = raw.select(*keys, F.posexplode("ItemsSold").alias("item_position", "i"))
item_discounts = item_rows.select(*keys, "item_position", F.posexplode("i.Discounts").alias("discount_position", "d"))

def role_events(frame, field, role_name):
    return frame.select(F.col(field).alias("employee_id"), F.lit(role_name).alias("role"))

roles = role_events(payments, "p.EmployeeId", "payment_actor")
for frame, field, role_name in [
    (check_discounts, "d.EmployeeId", "ticket_discount_actor"),
    (check_discounts, "d.ManagerId", "ticket_discount_approver"),
    (item_discounts, "d.EmployeeId", "item_discount_actor"),
    (item_discounts, "d.ManagerId", "item_discount_approver"),
]:
    roles = roles.unionByName(role_events(frame, field, role_name))
# A reused lazy DataFrame is not a materialized cache. Keep unioned source scans
# serial as well, so separate role branches do not allocate readers concurrently.
roles = roles.coalesce(1)
unique_employee_keys = (references["employees"].where("Id IS NOT NULL AND Id <> 0")
    .groupBy("Id").agg(F.count("*").alias("reference_records"))
    .where("reference_records = 1").select(F.col("Id").alias("matched_employee_id")))
role_summary = roles.join(F.broadcast(unique_employee_keys),
    roles.employee_id == unique_employee_keys.matched_employee_id, "left").groupBy("role").agg(
    F.count("*").alias("events"), F.count("employee_id").alias("id_present"),
    F.sum(F.when(F.col("employee_id") == 0, 1).otherwise(0)).alias("zero_id_events"),
    F.count("matched_employee_id").alias("uniquely_resolved_events"),
).orderBy("role")
emit("SAMPLE_ACTOR_REFERENCE_COUNTS", role_summary.collect())


## Inspect empirical money definitions

The production definitions use recorded line totals, without multiplying them by
`Quantity`. Item discounts are already reflected in `SoldPrice`.

- Net sales = sum of item sold prices minus ticket-level discounts.
- Recorded tip = minus the original payment `Gratuity`; original signs remain.
- Settlement difference = payment amount plus original gratuity minus
  (net sales plus tax plus service charge).
- Item-discount difference = gross item total minus sold item total minus
  item-discount events.

The tolerance of 0.05 is an empirical reconciliation setting, not an official
accounting rule. No currency is assumed. Missing arrays or monetary fields make a
ticket unevaluable; display totals based on zero are not observed zero values.
Negative payments, including gift-card-sale context, are retained. Summaries below
cover the chosen day only, and exceptions remain visible as counts and magnitudes.


In [ ]:
valid_items = F.forall("ItemsSold", lambda item:
    item.isNotNull() & item.GrossPrice.isNotNull() & item.SoldPrice.isNotNull()
    & item.Quantity.isNotNull() & item.Discounts.isNotNull()
    & F.forall(item.Discounts, lambda discount: discount.isNotNull() & discount.Amount.isNotNull()))
valid_payments = F.forall("Payments", lambda payment:
    payment.isNotNull() & payment.Amount.isNotNull() & payment.Gratuity.isNotNull())
valid_discounts = F.forall("Discounts", lambda discount: discount.isNotNull() & discount.Amount.isNotNull())
valid_taxes = F.forall("Tax", lambda tax: tax.isNotNull() & tax.Amount.isNotNull())
m = raw.select(
    F.coalesce(valid_items & valid_payments & valid_discounts & valid_taxes
               & F.col("ServiceCharge").isNotNull(), F.lit(False)).alias("money_complete"),
    F.coalesce(F.array_size("ItemsSold"), F.lit(0)).alias("item_count"),
    F.coalesce(F.array_size("Payments"), F.lit(0)).alias("payment_count"),
    array_money_total("ItemsSold", "GrossPrice").alias("gross_amount"),
    array_money_total("ItemsSold", "SoldPrice").alias("sold_amount"),
    array_money_total("Discounts", "Amount").alias("ticket_discount_amount"),
    array_money_total("Payments", "Amount").alias("payment_amount"),
    array_money_total("Payments", "Gratuity").alias("raw_gratuity_amount"),
    array_money_total("Tax", "Amount").alias("tax_amount"),
    F.col("ServiceCharge").alias("service_charge_amount"),
    F.expr("""coalesce(aggregate(ItemsSold, cast(0 as decimal(18,4)),
        (s,i) -> cast(s + coalesce(aggregate(i.Discounts, cast(0 as decimal(18,4)),
        (t,d) -> cast(t + coalesce(d.Amount,cast(0 as decimal(18,4))) as decimal(18,4))),
        cast(0 as decimal(18,4))) as decimal(18,4))), cast(0 as decimal(18,4)))""").alias("item_discount_amount"),
).withColumn("net_amount", money(F.col("sold_amount") - F.col("ticket_discount_amount")))  .withColumn("recorded_tip_amount", money(-F.col("raw_gratuity_amount")))  .withColumn("settlement_difference", money(F.col("payment_amount") + F.col("raw_gratuity_amount")
    - F.col("net_amount") - F.col("tax_amount") - F.col("service_charge_amount")))  .withColumn("item_discount_difference", money(F.col("gross_amount") - F.col("sold_amount") - F.col("item_discount_amount")))  .withColumn("settlement_evaluable", F.col("money_complete") & (F.col("item_count") > 0) & (F.col("payment_count") > 0))

emit("SAMPLE_MONETARY_RECONCILIATION", m.agg(
    F.count("*").alias("sample_tickets"),
    F.sum(F.when(~F.col("money_complete"), 1).otherwise(0)).alias("money_incomplete_tickets"),
    F.sum(F.when(F.col("settlement_evaluable"), 1).otherwise(0)).alias("settlement_evaluable_tickets"),
    F.sum(F.when(F.col("settlement_evaluable") & (F.abs("settlement_difference") > F.lit(TOLERANCE)), 1).otherwise(0)).alias("settlement_exceptions"),
    F.sum(F.when(F.col("money_complete") & (F.col("item_count") > 0)
                 & (F.abs("item_discount_difference") > F.lit(TOLERANCE)), 1).otherwise(0)).alias("item_discount_exceptions"),
    F.max(F.when(F.col("settlement_evaluable"), F.abs("settlement_difference"))).alias("maximum_absolute_settlement_difference"),
    *[F.sum(F.when(F.col("money_complete"), F.col(name))).alias("complete_" + name)
      for name in ["gross_amount", "net_amount", "ticket_discount_amount", "item_discount_amount", "recorded_tip_amount"]],
).collect())


In [ ]:
# These are grouped summaries, not individual ticket or employee rows.
emit("SAMPLE_PAYMENT_CATEGORIES", payments.groupBy("p.PayCatName").agg(
    F.count("*").alias("payment_events"),
    F.sum("p.Amount").alias("recorded_payment_amount"),
    F.sum("p.Gratuity").alias("original_gratuity_amount"),
    F.sum(F.when(F.col("p.Amount") < 0, 1).otherwise(0)).alias("negative_amount_events"),
    F.sum(F.when(F.col("p.Gratuity") > 0, 1).otherwise(0)).alias("positive_raw_gratuity_events"),
).orderBy(F.desc("payment_events"), "PayCatName").limit(30).collect())

discount_events = check_discounts.select(
    F.lit("ticket").alias("scope"), "d.DiscountName", "d.Amount"
).unionByName(item_discounts.select(
    F.lit("item").alias("scope"), "d.DiscountName", "d.Amount"
)).coalesce(1)
emit("SAMPLE_DISCOUNT_LABELS", discount_events.groupBy("scope", "DiscountName").agg(
    F.count("*").alias("discount_events"), F.sum("Amount").alias("recorded_discount_amount"),
    F.sum(F.when(F.col("Amount") < 0, 1).otherwise(0)).alias("negative_amount_events"),
).orderBy(F.desc("discount_events"), "scope", "DiscountName").limit(30).collect())
print("READ_ONLY_SOURCE_VALIDATION_FINISHED")


## Interpretation and next step

Successful inventory assertions establish exact metadata coverage. Reference
assertions establish unique present keys for the three parsed catalogs. Sample
outputs describe source structure, reference resolution, recorded signs and the
empirical reconciliation equations for one day; exceptions are observations to
investigate, not findings of fraud or policy violations.

This notebook does not prove complete-history reconciliation, pipeline dependency
execution, expectation enforcement, or reexecution stability. Use the deployed
pipeline's successful update, `gold_pipeline_quality`, its explained issue table,
and the separate acceptance checks for those conclusions. Review the operational
README before consuming tables, because materialized views do not form one global
transaction across an update. Preserve the original JSON and private data boundary.
